# BONUS: Change Data Feed & CLONE

Advanced Delta Lake features: row-level change tracking (CDF) and table cloning (Shallow vs Deep).

> **Note:** This notebook extends [03 — Delta Lake Fundamentals](03_delta_lake.ipynb). Run that notebook first to ensure `customers_delta` and other base tables exist.

## Setup

In [0]:
%run ../../setup/00_setup

### Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta

# Display user context
display(
    spark.createDataFrame([
        (CATALOG, BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA)
    ], ['catalog', 'bronze_schema', 'silver_schema', 'gold_schema'])
)

# Set catalog and schema as default
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {BRONZE_SCHEMA}")

## Change Data Feed (CDF)

Tracking row-level changes (INSERT, UPDATE, DELETE) in Delta tables. CDF enables incremental ETL pipelines by exposing `_change_type` metadata.

**Two terms are often confused in data engineering:**

### Change Data Capture (CDC)
A **pattern/technique** for capturing changes from source systems (databases, APIs, etc.)

| Aspect | Detail |
|--------|--------|
| **Scope** | Source-side technology |
| **What it does** | Captures INSERT, UPDATE, DELETE from operational databases |
| **Tools** | Debezium, AWS DMS, Fivetran, Qlik Replicate |
| **Output** | Stream of change events |

### Change Data Feed (CDF)
A **Delta Lake feature** that records row-level changes within Delta tables.

| Aspect | Detail |
|--------|--------|
| **Scope** | Delta Lake native feature |
| **What it does** | Tracks changes that happen WITHIN Delta tables |
| **Metadata columns** | `_change_type`, `_commit_version`, `_commit_timestamp` |
| **Use case** | Efficient incremental processing — read only changed rows |

> **Exam Note:** CDC captures changes FROM sources, CDF tracks changes WITHIN Delta tables. They are complementary — CDC feeds data into Delta, CDF enables efficient downstream processing.

### Example: Enabling Change Data Feed

**Objective:** Enable CDF on a Delta table to track all row-level changes

In [0]:
# Create a table with CDF enabled from the start
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo (
    user_id STRING,
    name STRING,
    email STRING,
    status STRING,
    updated_at TIMESTAMP
) 
USING DELTA
TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

print("Table created with Change Data Feed enabled")

In [0]:
# Verify CDF is enabled
properties = spark.sql(f"SHOW TBLPROPERTIES {CATALOG}.{BRONZE_SCHEMA}.cdf_demo")
display(properties.filter(F.col("key").like("%change%")))

### Example: Generating and Tracking Changes

**Objective:** Perform various DML operations and observe how CDF tracks each one

In [0]:
# INSERT initial data
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.cdf_demo VALUES
    ('U001', 'Alice', 'alice@example.com', 'active', current_timestamp()),
    ('U002', 'Bob', 'bob@example.com', 'active', current_timestamp()),
    ('U003', 'Charlie', 'charlie@example.com', 'active', current_timestamp())
""")
print("Version 1: Initial INSERT completed")

In [0]:
# UPDATE a record
spark.sql(f"""
UPDATE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo
SET status = 'premium', updated_at = current_timestamp()
WHERE user_id = 'U001'
""")
print("Version 2: UPDATE completed — Alice upgraded to premium")

In [0]:
# DELETE a record
spark.sql(f"""
DELETE FROM {CATALOG}.{BRONZE_SCHEMA}.cdf_demo
WHERE user_id = 'U002'
""")
print("Version 3: DELETE completed — Bob removed")

In [0]:
# INSERT new record
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.cdf_demo VALUES
    ('U004', 'Diana', 'diana@example.com', 'trial', current_timestamp())
""")
print("Version 4: INSERT completed — Diana added")

### Example: Reading Change Data Feed

**Objective:** Read and analyze change data with CDF metadata columns

In [0]:
# Current state of the table (standard read)
changes = spark.read \
    .format("delta") \
    .table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo")

display(changes)

In [0]:
# Read ALL changes from the beginning using CDF
changes = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo")

# Show changes with CDF metadata columns
display(
    changes.select(
        "user_id", "name", "status",
        "_change_type",        # insert, update_preimage, update_postimage, delete
        "_commit_version",     # Delta version number
        "_commit_timestamp"    # When the change occurred
    ).orderBy("_commit_version", "user_id")
)

**Understanding `_change_type` values:**

| Change Type | Description |
|-------------|-------------|
| `insert` | New row inserted |
| `update_preimage` | Row value **BEFORE** update |
| `update_postimage` | Row value **AFTER** update |
| `delete` | Row that was deleted |

> This enables powerful incremental processing patterns — you can process only what changed since the last pipeline run!

In [0]:
# Get only new inserts since version 2
new_inserts = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 2) \
    .table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo") \
    .filter(F.col("_change_type") == "insert")

print("New inserts since version 2:")
display(new_inserts.select("user_id", "name", "status", "_commit_version"))

In [0]:
# Get all deletions for audit purposes
deletions = spark.read \
    .format("delta") \
    .option("readChangeFeed", "true") \
    .option("startingVersion", 0) \
    .table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo") \
    .filter(F.col("_change_type") == "delete")

print("All deleted records (for audit):")
display(deletions.select("user_id", "name", "_commit_version", "_commit_timestamp"))

**RESTORE & CLONE — Syntax Reference**

| Operation | Syntax |
|-----------|--------|
| Restore to version | `RESTORE TABLE t TO VERSION AS OF 5` |
| Restore to timestamp | `RESTORE TABLE t TO TIMESTAMP AS OF '2024-01-01'` |
| Shallow Clone (instant) | `CREATE TABLE clone_t SHALLOW CLONE source_t` — metadata only, shares data files |
| Deep Clone (full copy) | `CREATE TABLE clone_t DEEP CLONE source_t` — independent copy of all data |
| Clone to version | `CREATE TABLE clone_t DEEP CLONE source_t VERSION AS OF 3` |

## CLONE — Shallow vs Deep Copy

Delta Lake supports two types of table cloning:

| Aspect | Shallow Clone | Deep Clone |
|--------|--------------|------------|
| **Data files** | References source files (no copy) | Full independent copy |
| **Speed** | Very fast (metadata only) | Slower (copies all data) |
| **Storage cost** | Minimal (shared files) | Full duplicate |
| **Independence** | Depends on source (VACUUM can break it) | Fully independent |
| **Use case** | Testing, short-lived experiments | Migration, backup, environment promotion |

```sql
-- Shallow Clone — fast, references source files
CREATE TABLE my_catalog.dev.customers_test
SHALLOW CLONE my_catalog.prod.customers;

-- Deep Clone — full independent copy
CREATE TABLE my_catalog.backup.customers_backup
DEEP CLONE my_catalog.prod.customers;

-- Clone a specific version (time travel + clone)
CREATE TABLE my_catalog.backup.customers_v5
DEEP CLONE my_catalog.prod.customers VERSION AS OF 5;
```

> **Pro Tip:** Shallow clones are perfect for creating test/dev copies of production tables — zero storage cost, instant creation. But remember: if the source table is VACUUMed, the shallow clone may break. For durable copies, use Deep Clone.

In [0]:
# --- Shallow Clone Demo ---
# Create a shallow clone of the CDF demo table (instant, no data copy)
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_shallow_clone
SHALLOW CLONE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo
""")

# Verify — same data, different table
original_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo").count()
clone_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo_shallow_clone").count()
print(f"Original: {original_count} rows | Shallow Clone: {clone_count} rows")

# Show clone metadata — note "cloneSource" in properties
display(spark.sql(f"DESCRIBE DETAIL {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_shallow_clone"))

In [0]:
# [1/3] Create deep clone — full independent copy (all data files physically duplicated)
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_deep_clone
DEEP CLONE {CATALOG}.{BRONZE_SCHEMA}.cdf_demo
""")

deep_count = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo_deep_clone").count()
print(f"Deep Clone created: {deep_count} rows (all data files physically copied)")

In [0]:
# [2/3] Modify clone — insert a row that does NOT exist in the source table
spark.sql(f"""
INSERT INTO {CATALOG}.{BRONZE_SCHEMA}.cdf_demo_deep_clone
VALUES ('U999', 'Clone-Only User', 'clone@test.com', 'test', current_timestamp())
""")
print("Inserted 'Clone-Only User' into deep clone — source table is untouched")

In [0]:
# [3/3] Verify independence — clone and source have diverged after the INSERT
orig = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo").count()
deep = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.cdf_demo_deep_clone").count()
print(f"Original:   {orig} rows")
print(f"Deep Clone: {deep} rows")
print(f"Extra rows in clone: {deep - orig} — DEEP CLONE does not share files with source")

← [03 — Delta Lake](03_delta_lake.ipynb) | **[ README](../../../README.md)**